In [0]:
# This notebook is the counterpart of our station_status and station_info downloaders. It is going to build a weather forecast archive
# that we can use to inform our predictions. These will be json files that will get processed into bronze (raw strings)
# and then later into silver. 

# Some background on what is happening with the weather forecasts before we proceed. Weather forecasts consist of model simulations.
# Each model simulation of the gfs-hrrr forecast predicts weather conditions over an 18 hour horizon. Each model run as an initialization time,
# which is the time at which the forecast "starts". This is different than the availability time, which will typically be several hours later.

# For each forecast that we download, we will save the entire forecast over the time horizon. 
# Here is a description of the json files:
# Archive HRRR forecasts for the locations in hrrr_locations.json.
# Each gzip JSON file contains one weather location and one model run.
#
# fetched_at: when we received the response, in UTC.
# forecast_initialized_at: the model run's initialization time, in UTC.
# location_id: weather cell identifier used by the station mapping.
# model, source, collection_type: forecast provenance.
# request_parameters: the API inputs used for this file.
# payload: the original response, including values, times, and units.

from datetime import datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import urlencode
import gzip
import json
import time
import urllib3
from urllib3.util import Retry, Timeout

# Keep development forecasts and configuration separate from production.
try:
    RUN_MODE = dbutils.widgets.get("run_mode")
except Exception:
    RUN_MODE = "dev"

if RUN_MODE not in ("dev", "production"):
    raise ValueError("run_mode must be 'dev' or 'production'")

WEATHER_ROOT = Path("/Volumes/citibike_project/citibike/raw/weather_forecast")
RUN_DIRECTORY = (
    WEATHER_ROOT if RUN_MODE == "production" else WEATHER_ROOT / "_dev"
)
ARCHIVE_ROOT = RUN_DIRECTORY / "hrrr"
API_URL = "https://single-runs-api.open-meteo.com/v1/forecast"


# These are the weather variables we are interested in

WEATHER_VARIABLES = [
    "temperature_2m", # Temperature and humidity at 2meters elevation. Temperature is in celsius
    "relative_humidity_2m",
    "precipitation",
    "snowfall", # cm/snow per hour
    "snow_depth", # meters of snow on ground
    "wind_speed_10m", # Wind speed at 10 meters height in m/s
    "cloud_cover", # Technically there could be enormous nuance in how this is measured, we keep it simple here
]

# We are going to target the four most extensive daily runs of the HRRR model, which occur at 00, 06, 12, and 18 UTC.
# see https://emc.ncep.noaa.gov/emc/pages/numerical_forecast_systems/hrrr.php
# These ones go out to 48 hours in advance. I doubt we need that level for the stockout prediction over a short time horizon,
# but maybe someone else will find them interesting. There could be several other ways to pick which runs to get

# There is the potential for some slop in the timing of forecast releases because the runs need to be completed, uploaded, and then distributed
# to open-meteo. We will basically look back in time to see which of the main runs is available. If it is there we get it, if not we wait
# until the next hour

# Load the saved weather locations and request settings.

configuration = json.loads(
    (RUN_DIRECTORY / "reference/hrrr_locations.json").read_text()
)
if not configuration["locations"]:
    raise ValueError("The weather location list is empty")

# Select the latest 00, 06, 12, or 18 UTC cycle
now = datetime.now(timezone.utc)
run_time = now.replace(
    hour=(now.hour // 6) * 6,
    minute=0,
    second=0,
    microsecond=0,
)

# Polite retries
http = urllib3.PoolManager(
    timeout=Timeout(connect=10, read=60),
    retries=Retry(
        total=3,
        backoff_factor=5,
        status_forcelist=[429, 500, 502, 503, 504], # These particular errors trigger a retry
        allowed_methods={"GET"},
        respect_retry_after_header=True, # They might tell us how long to wait
    ),
)

# We expect the forecast to cover the next 48 hours, and for the timestamps
# that we get from the forecast to be in thse formats
expected_times = [
    (run_time + timedelta(hours=hour)).strftime("%Y-%m-%dT%H:%M")
    for hour in range(49)
]

print(f"RUN_MODE = {RUN_MODE}")
print(f"Archive: {ARCHIVE_ROOT}")
print(f"Checking forecast run: {run_time.isoformat()}")

downloaded = skipped = 0

# Loop over locations

for location in configuration["locations"]:

    # Skip location/run combinations that are already archived. This way we can keep looking
    # without downloading and replacing the same thing alot. Just a note that we are doing this
    # inside the loop. In theory the hrrr will release one forecast which will cover all the locations
    # but it could be possible that a given location won't have access to that forecast at the same
    # time as the other locations, so we check if we have it
    output_path = (
        ARCHIVE_ROOT / location["location_id"]
        / run_time.strftime("year=%Y/month=%m/day=%d")
        / f"hrrr_{run_time:%Y%m%dT%H%M%SZ}.json.gz"
    )

    # We go to the next if the file is already there

    if output_path.exists():
        skipped += 1
        continue

    # Request this location's forecast using the saved grid settings.
    request_parameters = {
        "latitude": location["latitude"],
        "longitude": location["longitude"],
        "models": configuration["model"],
        **configuration["grid_options"],
        "run": run_time.strftime("%Y-%m-%dT%H:%M"),
        "hourly": ",".join(WEATHER_VARIABLES),
        "forecast_hours": 49,
        "timezone": "UTC",
        "temperature_unit": "celsius",
        "wind_speed_unit": "ms",
        "precipitation_unit": "mm",
    }

    # We send our request

    response = http.request(
        "GET", f"{API_URL}?{urlencode(request_parameters)}"
    )

    # An unpublished run is expected; leave it for the next scheduled execution.
    if response.status == 400:
        error_message = json.loads(response.data).get("reason", "")
        if error_message.startswith("The requested model run is not available"):
            print(f"Run {run_time.isoformat()} is not ready; trying again next execution.")
            break

    # Other errors should remain visible.
    if response.status != 200:
        raise RuntimeError(
            f"HTTP {response.status}: {response.data.decode()}"
        )

    payload = json.loads(response.data)
    fetched_at = datetime.now(timezone.utc).isoformat()

    # Confirm that the API selected the expected weather cell.
    # This type of comparison triggers my paranoia since it is comparing
    # things that originated as floats, albeit from same source.
    
    returned_id = (
        f"hrrr_{payload['latitude']:.6f}_{payload['longitude']:.6f}"
    )
    if returned_id != location["location_id"]:
        raise ValueError(
            f"Grid cell changed for {location['location_id']}: {returned_id}"
        )

    # Check completeness, allowing missing precipitation and snowfall at hour zero which commonly occurs
    hourly = payload["hourly"]
    if hourly["time"] != expected_times:
        raise ValueError(f"Unexpected forecast times for {run_time}")
    
    for variable in WEATHER_VARIABLES:
        values = hourly[variable]
        required_values = (
            values[1:]
            if variable in ("precipitation", "snowfall")
            else values
        )
        if len(values) != 49 or any(value is None for value in required_values):
            raise ValueError(
                f"Incomplete {variable}: {location['location_id']}, {run_time}"
            )

    # Preserve the original response, request parameters, and collection time.
    record = {
        "fetched_at": fetched_at,
        "forecast_initialized_at": run_time.isoformat(),
        "location_id": location["location_id"],
        "model": configuration["model"],
        "source": "open_meteo_single_runs",
        "collection_type": "scheduled_collection",
        "request_parameters": request_parameters,
        "payload": payload,
    }

    # Finish writing before placing the file it its ultimate location
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = output_path.with_suffix(".gz.tmp")

    with gzip.open(temporary_path, "wt", encoding="utf-8") as archive:
        json.dump(record, archive)

    temporary_path.replace(output_path)
    downloaded += 1
    time.sleep(1)

http.clear()
print(f"Downloaded: {downloaded}; already saved: {skipped}")

In [0]:
# Inspect the archived files for the run selected by the downloader.
checks = []
example_forecast = None

for location in configuration["locations"]:
    archive_path = (
        ARCHIVE_ROOT / location["location_id"]
        / run_time.strftime("year=%Y/month=%m/day=%d")
        / f"hrrr_{run_time:%Y%m%dT%H%M%SZ}.json.gz"
    )

    if not archive_path.exists():
        checks.append({
            "location_id": location["location_id"],
            "saved": False,
        })
        continue

    # Reading the complete gzip file also checks that it can be decompressed.
    with gzip.open(archive_path, "rt", encoding="utf-8") as archive:
        record = json.load(archive)

    hourly = record["payload"]["hourly"]
    returned_id = (
        f"hrrr_{record['payload']['latitude']:.6f}_"
        f"{record['payload']['longitude']:.6f}"
    )

    checks.append({
        "location_id": location["location_id"],
        "saved": True,
        "location_matches": (
            record["location_id"] == location["location_id"] == returned_id
        ),
        "run_matches": (
            datetime.fromisoformat(record["forecast_initialized_at"]) == run_time
        ),
        "times_match": hourly["time"] == expected_times,
        "hours": len(hourly["time"]),
        "fetched_at": record["fetched_at"],
        "compressed_bytes": archive_path.stat().st_size,
        **{
            f"{variable}_missing": sum(
                value is None for value in hourly[variable]
            )
            for variable in WEATHER_VARIABLES
        },
    })

    if example_forecast is None:
        example_forecast = record

# Display archive coverage and any missing files.
import pandas as pd

archive_checks = pd.DataFrame(checks)
saved_count = int(archive_checks["saved"].sum())

print(f"Forecast run: {run_time.isoformat()}")
print(f"Saved locations: {saved_count}/{len(configuration['locations'])}")
display(archive_checks)

# Display one location's units and complete hourly forecast.
if example_forecast is not None:
    print(f"Example location: {example_forecast['location_id']}")
    print(f"Units: {example_forecast['payload']['hourly_units']}")
    display(pd.DataFrame(example_forecast["payload"]["hourly"]))